# 8.11 — Time-resolved Decoding with 1-Layer FC Network

Loads the pseudo-population dataset prepared by **8.10** and fits a
single fully-connected (linear) layer with **50% input dropout** for each
50 ms time bin.  Null distribution is computed by shuffling training labels.

## Architecture: `OneLayerFC`
```
Input (n_neurons)  →  Dropout(0.5)  →  Linear(n_neurons, n_classes)
```
One FC layer, input dropout regularisation, CrossEntropyLoss, Adam optimiser.
Input spike counts are z-scored per neuron (fit on training set only).

**Prerequisites**: run `8.10-decoding-data-preparation.ipynb` first.

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from imports import *
from config import dir_config, ephys_config
from src.utils.decoding_utils import get_label_pseudo_trials, get_binned_counts

In [ ]:
processed_dir = Path(dir_config.data.processed)
GP_EPHYS_CFG  = ephys_config["alignment_settings_GP"]
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

load_path = processed_dir / "session_trial_data.pkl"
with open(load_path, "rb") as f:
    saved = pickle.load(f)

session_data     = saved["session_data"]      # {event: {session_id: {spikes, coherence, choice, hmm_state}}}
neuron_positions = saved["neuron_positions"]  # {session_id: global_neuron_indices}
n_total_neurons  = saved["n_total_neurons"]
label_values     = saved["label_values"]      # {"coherence": ndarray, "choice": ndarray, "hmm_state": ndarray}

alignments = list(session_data.keys())
print(f"\nLoaded: {load_path}")
print(f"Alignments     : {alignments}")
print(f"Total neurons  : {n_total_neurons}")
for event, event_data in session_data.items():
    n_sess   = len(event_data)
    n_trials = sum(v["spikes"].shape[0] for v in event_data.values())
    print(f"  [{event}]: {n_sess} sessions, {n_trials} total trials")

In [ ]:
decode_cfg = {
    "coherence": {
        "label_col":    "coherence",
        "label_values": label_values["coherence"],
        "n_classes":    len(label_values["coherence"]),
        "label_names":  [str(v) for v in label_values["coherence"]],
    },
    "choice": {
        "label_col":    "choice",
        "label_values": label_values["choice"],
        "n_classes":    len(label_values["choice"]),
        "label_names":  [str(v) for v in label_values["choice"]],
    },
    "hmm_state": {
        "label_col":    "hmm_state",
        "label_values": label_values["hmm_state"],
        "n_classes":    len(label_values["hmm_state"]),
        "label_names":  [str(v) for v in label_values["hmm_state"]],
    },
}

# ── Hyperparameters ───────────────────────────────────────────────────────────
N_TRAIN   = 800   # bootstrap pseudo-trials per label value (train)
N_TEST    = 200   # bootstrap pseudo-trials per label value (test)
N_REPEATS = 10    # independent 80/20 splits
TEST_SIZE = 0.2   # fraction of each session's trials held out
BIN_SIZE  = 50    # ms per time bin
BIN_STEP  = 25    # ms between bin starts (overlapping)
N_EPOCHS  = 200   # Adam epochs per model fit
LR        = 1e-3  # Adam learning rate
DROPOUT   = 0.5   # input dropout probability

for lc, cfg in decode_cfg.items():
    chance = 1 / cfg["n_classes"]
    print(f"{lc:<12}  n_classes={cfg['n_classes']}  chance={chance:.3f}  labels={cfg['label_names']}")

## Model definition

In [5]:
class OneLayerFC(nn.Module):
    """Single FC layer with input dropout — a dropout-regularised linear classifier."""

    def __init__(self, n_input: int, n_classes: int, dropout: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(n_input, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def fit_1fc(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test:  np.ndarray, y_test:  np.ndarray,
    n_classes: int,
    n_epochs: int   = 200,
    lr:       float = 1e-3,
    dropout:  float = 0.5,
    device:   str   = "cpu",
) -> float:
    """
    Z-score X (fit on train), train OneLayerFC, return test accuracy.

    y_train / y_test must be integer class indices (0..n_classes-1).
    """
    mu  = X_train.mean(axis=0, keepdims=True)
    sig = X_train.std(axis=0,  keepdims=True) + 1e-8
    X_tr_sc = (X_train - mu) / sig
    X_te_sc = (X_test  - mu) / sig

    X_tr = torch.tensor(X_tr_sc, dtype=torch.float32, device=device)
    y_tr = torch.tensor(y_train,  dtype=torch.long,    device=device)
    X_te = torch.tensor(X_te_sc, dtype=torch.float32, device=device)
    y_te = torch.tensor(y_test,   dtype=torch.long,    device=device)

    model     = OneLayerFC(X_train.shape[1], n_classes, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(n_epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        acc = (model(X_te).argmax(dim=1) == y_te).float().mean().item()
    return acc

## Decoding function

In [ ]:
def compute_bin_edges(alignment, bin_size=50, bin_step=25):
    """Return (bin_edges, bin_centres_ms) for a given alignment epoch."""
    cfg      = GP_EPHYS_CFG[alignment]
    start_ms = cfg["start_time_ms"]
    end_ms   = cfg["end_time_ms"]
    n_total  = end_ms - start_ms + 1
    edges    = [(t, t + bin_size) for t in range(0, n_total - bin_size + 1, bin_step)]
    centres  = [start_ms + t + bin_size // 2 for t, _ in edges]
    return edges, centres


def decode_pseudo_population(
    session_data_event,
    label_col,
    label_values,
    neuron_positions,
    n_total_neurons,
    n_train   = 800,
    n_test    = 200,
    n_repeats = 10,
    test_size = 0.2,
    bin_edges = None,
    n_epochs  = 200,
    lr        = 1e-3,
    dropout   = 0.5,
    device    = "cpu",
):
    """Time-resolved 1FC decoding with per-session independent bootstrapping.

    For each repeat:
      1. Per session, per label value: 80/20 split, then sample n_train/n_test indices.
      2. For each time bin: stack sessions neuron-wise into pseudo-population X; fit 1FC.

    Each session independently picks trials for a given label value, so no session needs
    to have trials at the same slot as another — eliminates NaN sparsity.

    Parameters
    ----------
    session_data_event : {session_id: {"spikes": (n_trials, n_neurons, n_timebins),
                                        "coherence": ..., "choice": ..., "hmm_state": ...}}
    label_col          : which key in session_data_event to decode ("coherence" / "choice" / "hmm_state")
    label_values       : ordered array of unique label values
    neuron_positions   : {session_id: global_neuron_indices}
    n_total_neurons    : total neurons in pseudo-population

    Returns
    -------
    repeat_accs : (n_repeats, n_bins)
    mean_acc    : (n_bins,)
    sem_acc     : (n_bins,)
    """
    n_classes = len(label_values)
    n_bins    = len(bin_edges)

    # ── Diagnostic ────────────────────────────────────────────────────────────
    for v_idx, v in enumerate(label_values):
        counts = [(data[label_col] == v).sum() for data in session_data_event.values()]
        total  = sum(counts)
        n_sess = sum(c > 0 for c in counts)
        flag   = "  *** LOW ***" if total < 10 else ""
        print(f"  {label_col}={v!s:>7}: {total:4d} trials across {n_sess} sessions{flag}")
    print()

    repeat_accs = np.zeros((n_repeats, n_bins))

    for ri in range(n_repeats):
        rng = np.random.default_rng(seed=ri)

        # Pre-compute sampling indices for each (label_value, session) pair
        train_idx_map = {}
        test_idx_map  = {}

        for v_idx, v in enumerate(label_values):
            for session_id, data in session_data_event.items():
                mask = data[label_col] == v
                n_v  = mask.sum()

                if n_v == 0:
                    train_idx_map[(v_idx, session_id)] = None
                    test_idx_map[ (v_idx, session_id)] = None
                    continue

                perm     = rng.permutation(n_v)
                n_test_v = max(1, int(n_v * test_size))

                if n_test_v >= n_v:
                    if ri == 0:
                        print(f"  WARNING: session {session_id}, {label_col}={v!r} "
                              f"has only {n_v} trials — resampling all for train and test.")
                    tr_idx = rng.choice(n_v, n_train, replace=True)
                    te_idx = rng.choice(n_v, n_test,  replace=True)
                else:
                    te_pool = perm[:n_test_v]
                    tr_pool = perm[n_test_v:]
                    tr_idx  = rng.choice(tr_pool, n_train, replace=True)
                    te_idx  = rng.choice(te_pool, n_test,  replace=True)

                train_idx_map[(v_idx, session_id)] = tr_idx
                test_idx_map[ (v_idx, session_id)] = te_idx

        # ── Per time bin ──────────────────────────────────────────────────────
        for bi, (t0, t1) in enumerate(bin_edges):
            X_tr = np.zeros((n_classes * n_train, n_total_neurons))
            X_te = np.zeros((n_classes * n_test,  n_total_neurons))

            for v_idx, v in enumerate(label_values):
                tr_start = v_idx * n_train
                te_start = v_idx * n_test

                for session_id, data in session_data_event.items():
                    tr_idx = train_idx_map[(v_idx, session_id)]
                    te_idx = test_idx_map[ (v_idx, session_id)]
                    if tr_idx is None:
                        continue

                    mask   = data[label_col] == v
                    binned = data["spikes"][mask, :, t0:t1].sum(-1)  # (n_v, n_session_neurons)
                    npos   = neuron_positions[session_id]

                    X_tr[tr_start:tr_start + n_train, npos] = binned[tr_idx]
                    X_te[te_start:te_start + n_test,  npos] = binned[te_idx]

            y_tr = np.repeat(np.arange(n_classes), n_train)
            y_te = np.repeat(np.arange(n_classes), n_test)

            repeat_accs[ri, bi] = fit_1fc(
                X_tr, y_tr, X_te, y_te,
                n_classes=n_classes, n_epochs=n_epochs,
                lr=lr, dropout=dropout, device=device,
            )

        print(f"  repeat {ri + 1}/{n_repeats}", end="\r")
    print()

    mean_acc = repeat_accs.mean(axis=0)
    sem_acc  = repeat_accs.std(axis=0) / np.sqrt(n_repeats)
    return repeat_accs, mean_acc, sem_acc

## Run decoding

In [ ]:
results = {}
dev     = str(device)

for label_col, cfg in decode_cfg.items():
    results[label_col] = {}
    chance = 1 / cfg["n_classes"]
    fig, axes = plt.subplots(1, len(alignments), figsize=(14, 3), sharey=True)

    for ax, alignment in zip(axes, alignments):
        if alignment not in session_data:
            ax.set_visible(False)
            continue

        print(f"\n{label_col} × {alignment}")
        bin_edges, bin_centres_ms = compute_bin_edges(alignment, bin_size=BIN_SIZE, bin_step=BIN_STEP)

        repeat_accs, mean_acc, sem_acc = decode_pseudo_population(
            session_data[alignment],
            label_col       = cfg["label_col"],
            label_values    = cfg["label_values"],
            neuron_positions = neuron_positions,
            n_total_neurons  = n_total_neurons,
            n_train   = N_TRAIN,
            n_test    = N_TEST,
            n_repeats = N_REPEATS,
            test_size = TEST_SIZE,
            bin_edges = bin_edges,
            n_epochs  = N_EPOCHS,
            lr        = LR,
            dropout   = DROPOUT,
            device    = dev,
        )

        results[label_col][alignment] = {
            "repeat_accs": repeat_accs,
            "mean_acc":    mean_acc,
            "sem_acc":     sem_acc,
            "bin_centres": bin_centres_ms,
        }

        ax.plot(bin_centres_ms, mean_acc, color="steelblue", label="1FC")
        ax.fill_between(bin_centres_ms, mean_acc - sem_acc, mean_acc + sem_acc,
                        alpha=0.3, color="steelblue")
        ax.axhline(chance, color="k",    linestyle=":", linewidth=0.8, label="chance")
        ax.axvline(0,      color="gray", linestyle=":", linewidth=0.8)
        ax.set_title(alignment)
        ax.set_xlabel("Time from event (ms)")

    axes[0].set_ylabel("Decoding accuracy")
    axes[-1].legend(fontsize=7, loc="lower right")
    fig.suptitle(f"Decoding: {label_col}  (chance={chance:.2f})  —  1FC + {int(DROPOUT*100)}% dropout")
    plt.tight_layout()
    plt.show()

# ── Save ─────────────────────────────────────────────────────────────────────
save_path = processed_dir / "decoding_results_1fc.pkl"
with open(save_path, "wb") as f:
    pickle.dump(results, f)
print(f"\nSaved: {save_path}")